# Parameter Grid Tester — Quick Config Testing

Test different TP/SL/K1/K2/stability combos across coins without retraining.

**Key idea:** Load pre-trained models + features, swap parameter grids, evaluate PF/trades, compare.

**Inputs:**
- `TEST_COINS`: Which coins to test
- `PARAM_CONFIGS`: List of parameter grid configs to test
- `MIN_PF_GATE`, `PF_STABILITY_RATIO_GATE`: Validity thresholds

**Output:** Comparison table + summary stats

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from pathlib import Path

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import config
from utils.labeling import generate_labels
from utils.model_utils import train_xgboost, compute_profit_factor
from utils.data_utils import load_snapshot_csv

print(f'Config loaded. Tickers: {len(config.CRYPTO_TICKERS)}, Timeframe: {config.TIMEFRAME}')

## 1. Setup: Define coins and parameter grids

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# CONFIG: Test coins
# ────────────────────────────────────────────────────────────────────────────

TEST_COINS = ['XRP/USDT', 'SOL/USDT', 'LTC/USDT', 'ADA/USDT', 'AAVE/USDT',
    'LINK/USDT', 'AVAX/USDT', 'TRX/USDT', 'FIL/USDT', 'BCH/USDT', 'ZEC/USDT']  # Edit this

# ────────────────────────────────────────────────────────────────────────────
# CONFIG: Parameter grid configs to test
# Each entry: (name, tp_grid, sl_grid, k1_grid, k2_grid, min_pf, pf_stability)
# ────────────────────────────────────────────────────────────────────────────

PARAM_CONFIGS = [
    (
        'Baseline (current)',
        [0.007],                           # TP 0.03
        [0.004],                           # SL 0.01
        [0.5],                             # K1
        [0.3],                             # K2
        1.20,                              # MIN_PF_TEST1
        0.65,                              # PF_STABILITY_RATIO
    ),
    (
        'Tight (smaller grid)',
        [0.0035, 0.005, 0.008],             # TP
        [0.002, 0.0025, 0.003],             # SL
        [0.3, 0.5],                        # K1
        [0.2, 0.3],                        # K2
        1.20,                              # MIN_PF_TEST1
        0.70,                              # PF_STABILITY_RATIO
    ),
    (
        'Wide (larger grid)',
        [0.010, 0.015, 0.020, 0.025, 0.030],  # TP
        [0.006, 0.01, 0.015, 0.02],         # SL
        [0.3, 0.5, 0.7],                      # K1
        [0.2, 0.3, 0.4],                      # K2
        1.20,                              # MIN_PF_TEST1
        0.70,                              # PF_STABILITY_RATIO
    ),
    (
        'Relaxed stability',
        [0.030],                           # TP
        [0.015],                           # SL
        [0.5],                             # K1
        [0.3],                             # K2
        1.05,                              # MIN_PF_TEST1 (lower)
        0.50,                              # PF_STABILITY_RATIO (lower)
    ),
]

print(f'Testing {len(PARAM_CONFIGS)} configs on {len(TEST_COINS)} coins:\n')
for name, tp, sl, k1, k2, min_pf, pf_ratio in PARAM_CONFIGS:
    grid_size = len(tp) * len(sl) * len(k1) * len(k2)
    print(f'  {name:30s} | Grid: {grid_size:3d} | PF_MIN: {min_pf:.2f} | Stability: {pf_ratio:.2f}')

## 2. Load data + split into train/test1/test2

In [ ]:
def load_coin_data(ticker):
    """Load CSV snapshot for a coin, return full OHLCV + ATR."""
    csv_path = os.path.join(config.DATA_WORKING, 'crypto', f'{ticker.replace("/", "_")}_1h_snapshot.csv')
    if not os.path.exists(csv_path):
        print(f'  ⚠️  {ticker}: snapshot not found at {csv_path}')
        return None
    
    df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
    df.index.name = 'timestamp'
    return df

# Load data
data = {}
print('Loading data...')
for ticker in TEST_COINS:
    df = load_coin_data(ticker)
    if df is not None:
        data[ticker] = df
        print(f'  ✓ {ticker:15s} | {len(df):5d} rows | {df.index.min():%Y-%m-%d} to {df.index.max():%Y-%m-%d}')

print(f'\nLoaded {len(data)}/{len(TEST_COINS)} coins')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# DEBUG: Inspect timezone + CSV content
# ════════════════════════════════════════════════════════════════════════════

print('\n' + '='*80)
print('DEBUG: TIMEZONE INVESTIGATION')
print('='*80 + '\n')

# Step 1: Check loaded data timezone
print('STEP 1: Inspect Loaded Data Timezone\n')
for ticker, df in data.items():
    print(f'{ticker}:')
    print(f'  Index type: {type(df.index)}')
    print(f'  Index tz: {df.index.tz}')
    print(f'  Index dtype: {df.index.dtype}')
    print(f'  First 3 timestamps: {df.index[:3].tolist()}')
    print(f'  Sample value: {df.index[0]} (type: {type(df.index[0])})')
    print()

# Step 2: Test timestamp creation
print('\nSTEP 2: Test Timestamp Creation\n')
df_sample = data[list(data.keys())[0]]
print(f'Using sample ticker: {list(data.keys())[0]}')
print(f'DataFrame index tz: {df_sample.index.tz}\n')

ts_naive = pd.Timestamp.now()
ts_utc = pd.Timestamp.now(tz='UTC')
ts_from_index = df_sample.index[0]

print(f'Created naive: {ts_naive} (tz: {ts_naive.tz})')
print(f'Created UTC: {ts_utc} (tz: {ts_utc.tz})')
print(f'From index: {ts_from_index} (tz: {ts_from_index.tz})')

# Test comparison
print(f'\nTest comparisons:')
try:
    result = df_sample.index[0] <= ts_utc
    print(f'  ✓ Comparison df.index[0] <= ts_utc works: {result}')
except TypeError as e:
    print(f'  ✗ Comparison failed: {type(e).__name__}: {str(e)[:100]}')

try:
    result = df_sample.index[0] <= ts_naive
    print(f'  ✓ Comparison df.index[0] <= ts_naive works: {result}')
except TypeError as e:
    print(f'  ✗ Comparison failed: {type(e).__name__}: {str(e)[:100]}')

# Step 3: Check CSV file content
print(f'\n\nSTEP 3: Check Raw CSV Content\n')
import csv
for ticker in list(data.keys())[:1]:  # Just first coin to save output
    csv_path = os.path.join(config.DATA_WORKING, 'crypto', f'{ticker.replace("/", "_")}_1h_snapshot.csv')
    print(f'Reading: {csv_path}\n')
    with open(csv_path, 'r') as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            if i < 6:
                print(f'  Row {i}: {row[0][:50]}...' if len(row[0]) > 50 else f'  Row {i}: {row[0]}')
            elif i == 6:
                print(f'  ... (sample complete)')
                break

print('\n' + '='*80)

In [ ]:
def split_data_by_days(df, train_end_days=85, test1_range=(85, 31), test2_range=(31, 1)):
    """
    Split DataFrame by days from today.
    
    train_end_days: training ends N days ago
    test1_range: (start_days_ago, end_days_ago) → test set 1
    test2_range: (start_days_ago, end_days_ago) → test set 2
    """
    # FIX: Extract timezone from data, use it to create matching timestamp
    tz = df.index.tz if df.index.tz is not None else 'UTC'
    today = pd.Timestamp.now(tz=tz)
    
    train_end = today - pd.Timedelta(days=train_end_days)
    test1_start = today - pd.Timedelta(days=test1_range[0])
    test1_end = today - pd.Timedelta(days=test1_range[1])
    test2_start = today - pd.Timedelta(days=test2_range[0])
    test2_end = today - pd.Timedelta(days=test2_range[1])
    
    train = df[df.index <= train_end]
    test1 = df[(df.index > test1_start) & (df.index <= test1_end)]
    test2 = df[(df.index > test2_start) & (df.index <= test2_end)]
    
    return train, test1, test2

# Split data
splits = {}
print('Splitting data into train/test1/test2...\n')
for ticker, df in data.items():
    train, test1, test2 = split_data_by_days(
        df,
        train_end_days=config.TRAIN_END_DAYS,
        test1_range=config.TEST1,
        test2_range=config.TEST2,
    )
    splits[ticker] = {'train': train, 'test1': test1, 'test2': test2}
    print(f'{ticker:15s} | Train: {len(train):5d} | Test1: {len(test1):4d} | Test2: {len(test2):4d}')
    if len(train) > 0:
        print(f'             | {train.index.min():%Y-%m-%d} → {train.index.max():%Y-%m-%d} | '
              f'{test1.index.min():%Y-%m-%d}→{test1.index.max():%Y-%m-%d} | '
              f'{test2.index.min():%Y-%m-%d}→{test2.index.max():%Y-%m-%d}\n')
    else:
        print(f'             | (no data)\n')

## 3. Evaluation function: test single config on single coin

In [ ]:
def get_feature_cols(df):
    """Extract feature columns (exclude OHLCV)."""
    exclude = ['open', 'high', 'low', 'close', 'volume', 'ATR_14']
    return [c for c in df.columns if c not in exclude]

def eval_config(
    ticker,
    train,
    test1,
    test2,
    tp_grid,
    sl_grid,
    k1_grid,
    k2_grid,
    direction='long',
    verbose=False,
):
    """
    Evaluate a parameter grid on train/test1/test2.
    
    Returns dict with PF/trades for each test set and validity gates.
    """
    feature_cols = get_feature_cols(train)
    
    # Add label buffer to avoid look-ahead
    label_buffer = pd.Timedelta(hours=config.LABEL_HORIZON)
    train_clean = train[train.index <= (train.index.max() - label_buffer)]
    
    # Grid search: find best (tp, sl, k1, k2) on train
    best_score = -1
    best_params = None
    best_pf_train = 0
    best_n_train = 0
    
    from itertools import product
    grid = list(product(tp_grid, sl_grid, k1_grid, k2_grid))
    
    for tp, sl, k1, k2 in grid:
        # Generate labels on clean train
        labels = generate_labels(train_clean, tp, sl, k1, k2, direction=direction)
        valid_mask = labels.notna()
        X_train = train_clean.loc[valid_mask, feature_cols].copy()
        y_train = labels[valid_mask].astype(int)
        
        if len(y_train) < 30 or y_train.sum() < 5 or (len(y_train) - y_train.sum()) < 5:
            continue
        
        try:
            model = train_xgboost(X_train, y_train, n_estimators=config.GRID_SEARCH_ESTIMATORS, nthread=1)
        except Exception as e:
            if verbose:
                print(f'    Train failed for tp={tp} sl={sl} k1={k1} k2={k2}: {e}')
            continue
        
        # Score on train
        threshold = config.LONG_THRESHOLD if direction == 'long' else config.SHORT_THRESHOLD
        pf, n_trades = compute_profit_factor(
            model, X_train, y_train, threshold, direction=direction,
            tp_pct=tp, sl_pct=sl, k1=k1, k2=k2,
            atr_norm=X_train['ATR_14_norm'].values if 'ATR_14_norm' in X_train.columns else None,
        )
        score = pf * np.sqrt(n_trades) if n_trades >= config.MIN_TRADE_COUNT else -1
        
        if score > best_score:
            best_score = score
            best_params = (tp, sl, k1, k2)
            best_pf_train = pf
            best_n_train = n_trades
            best_model = model
    
    if best_params is None:
        return {'valid': False, 'reason': 'No valid combo found'}
    
    tp, sl, k1, k2 = best_params
    
    # Evaluate on test1
    labels_test1 = generate_labels(test1, tp, sl, k1, k2, direction=direction)
    valid_mask_test1 = labels_test1.notna()
    X_test1 = test1.loc[valid_mask_test1, feature_cols].copy()
    y_test1 = labels_test1[valid_mask_test1].astype(int)
    
    if len(y_test1) == 0:
        return {'valid': False, 'reason': 'No labels in test1'}
    
    pf_test1, n_test1 = compute_profit_factor(
        best_model, X_test1, y_test1, threshold, direction=direction,
        tp_pct=tp, sl_pct=sl, k1=k1, k2=k2,
        atr_norm=X_test1['ATR_14_norm'].values if 'ATR_14_norm' in X_test1.columns else None,
    )
    
    # Evaluate on test2
    labels_test2 = generate_labels(test2, tp, sl, k1, k2, direction=direction)
    valid_mask_test2 = labels_test2.notna()
    X_test2 = test2.loc[valid_mask_test2, feature_cols].copy()
    y_test2 = labels_test2[valid_mask_test2].astype(int)
    
    if len(y_test2) == 0:
        pf_test2, n_test2 = 0, 0
    else:
        pf_test2, n_test2 = compute_profit_factor(
            best_model, X_test2, y_test2, threshold, direction=direction,
            tp_pct=tp, sl_pct=sl, k1=k1, k2=k2,
            atr_norm=X_test2['ATR_14_norm'].values if 'ATR_14_norm' in X_test2.columns else None,
        )
    
    return {
        'valid': True,
        'tp': tp,
        'sl': sl,
        'k1': k1,
        'k2': k2,
        'pf_train': round(best_pf_train, 3),
        'n_train': best_n_train,
        'pf_test1': round(pf_test1, 3),
        'n_test1': n_test1,
        'pf_test2': round(pf_test2, 3),
        'n_test2': n_test2,
    }

print('eval_config() ready')

## 4. Run all configs on all coins

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Main evaluation loop
# ════════════════════════════════════════════════════════════════════════════

results = {}  # config_name -> {ticker -> result}

for config_name, tp_grid, sl_grid, k1_grid, k2_grid, min_pf, pf_ratio in PARAM_CONFIGS:
    print(f'\n{"="*80}')
    print(f'Config: {config_name}')
    print(f'Grid size: {len(tp_grid)} × {len(sl_grid)} × {len(k1_grid)} × {len(k2_grid)} = {len(tp_grid)*len(sl_grid)*len(k1_grid)*len(k2_grid)} combos')
    print(f'Gates: MIN_PF_TEST1={min_pf:.2f}, PF_STABILITY_RATIO={pf_ratio:.2f}')
    print(f'{"="*80}\n')
    
    results[config_name] = {}
    
    for ticker in TEST_COINS:
        if ticker not in splits:
            print(f'{ticker:15s} | SKIPPED (not loaded)')
            continue
        
        train, test1, test2 = splits[ticker]['train'], splits[ticker]['test1'], splits[ticker]['test2']
        
        result = eval_config(
            ticker,
            train,
            test1,
            test2,
            tp_grid,
            sl_grid,
            k1_grid,
            k2_grid,
            direction='long',
        )
        
        results[config_name][ticker] = result
        
        if not result['valid']:
            print(f'{ticker:15s} | ✗ {result["reason"]}')
        else:
            pf_test1 = result['pf_test1']
            n_test1 = result['n_test1']
            pf_test2 = result['pf_test2']
            n_test2 = result['n_test2']
            
            stability = pf_test2 / pf_test1 if pf_test1 > 0 else 0
            passes_pf = pf_test1 >= min_pf
            passes_stability = stability >= pf_ratio
            
            status = '✓' if (passes_pf and passes_stability) else '✗'
            print(f'{ticker:15s} | {status} PF_test1={pf_test1:.3f} ({n_test1} trades) | '
                  f'PF_test2={pf_test2:.3f} ({n_test2} trades) | Stability={stability:.2f}  '
                  f'[PF_OK:{passes_pf} Stab_OK:{passes_stability}]')

print(f'\n{"="*80}\nEvaluation complete!\n')

## 5. Comparison reports

In [ ]:
# Build comparison DataFrame per coin

def build_comparison_table(results_dict, config_names, min_pf_gates, pf_ratio_gates):
    """
    results_dict: {config_name -> {ticker -> result}}
    config_names: list of config names
    min_pf_gates: dict {config_name -> min_pf}
    pf_ratio_gates: dict {config_name -> pf_ratio}
    """
    rows = []
    
    for config_name in config_names:
        for ticker, result in results_dict[config_name].items():
            if not result['valid']:
                status = '✗ Invalid'
            else:
                min_pf = min_pf_gates[config_name]
                pf_ratio = pf_ratio_gates[config_name]
                pf_test1 = result['pf_test1']
                pf_test2 = result['pf_test2']
                stability = pf_test2 / pf_test1 if pf_test1 > 0 else 0
                
                passes_pf = pf_test1 >= min_pf
                passes_stability = stability >= pf_ratio
                status = '✓ PASS' if (passes_pf and passes_stability) else '✗ FAIL'
            
            rows.append({
                'Config': config_name,
                'Ticker': ticker,
                'PF_Test1': result.get('pf_test1', np.nan),
                'N_Test1': result.get('n_test1', 0),
                'PF_Test2': result.get('pf_test2', np.nan),
                'N_Test2': result.get('n_test2', 0),
                'Stability': (result.get('pf_test2', 0) / result.get('pf_test1', 1)) if result.get('pf_test1', 0) > 0 else 0,
                'Status': status,
            })
    
    return pd.DataFrame(rows)

# Build min_pf and pf_ratio dicts
min_pf_gates = {}
pf_ratio_gates = {}
for config_name, tp_grid, sl_grid, k1_grid, k2_grid, min_pf, pf_ratio in PARAM_CONFIGS:
    min_pf_gates[config_name] = min_pf
    pf_ratio_gates[config_name] = pf_ratio

# Build comparison table
config_names = [name for name, _, _, _, _, _, _ in PARAM_CONFIGS]
comparison_df = build_comparison_table(results, config_names, min_pf_gates, pf_ratio_gates)

print('\n' + '='*120)
print('FULL COMPARISON TABLE')
print('='*120)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(comparison_df.to_string(index=False))

In [ ]:
# Summary: Pass rate per config

print('\n' + '='*80)
print('SUMMARY: Pass rate by config')
print('='*80 + '\n')

for config_name in config_names:
    config_df = comparison_df[comparison_df['Config'] == config_name]
    pass_count = (config_df['Status'] == '✓ PASS').sum()
    total_count = len(config_df)
    pass_rate = pass_count / total_count if total_count > 0 else 0
    
    avg_pf_test1 = config_df['PF_Test1'].mean()
    avg_pf_test2 = config_df['PF_Test2'].mean()
    avg_stability = config_df['Stability'].mean()
    
    print(f'{config_name:30s} | Pass: {pass_count}/{total_count} ({pass_rate*100:5.1f}%) | '
          f'Avg PF_T1: {avg_pf_test1:.3f} | Avg PF_T2: {avg_pf_test2:.3f} | Avg Stab: {avg_stability:.2f}')

In [ ]:
# Per-coin summary

print('\n' + '='*80)
print('SUMMARY: Performance by coin')
print('='*80 + '\n')

for ticker in TEST_COINS:
    ticker_df = comparison_df[comparison_df['Ticker'] == ticker]
    if len(ticker_df) == 0:
        print(f'{ticker:15s} | No data')
        continue
    
    pass_count = (ticker_df['Status'] == '✓ PASS').sum()
    total_count = len(ticker_df)
    pass_rate = pass_count / total_count if total_count > 0 else 0
    
    avg_pf_test1 = ticker_df['PF_Test1'].mean()
    avg_pf_test2 = ticker_df['PF_Test2'].mean()
    avg_stability = ticker_df['Stability'].mean()
    
    print(f'{ticker:15s} | Pass: {pass_count}/{total_count} ({pass_rate*100:5.1f}%) | '
          f'Avg PF_T1: {avg_pf_test1:.3f} | Avg PF_T2: {avg_pf_test2:.3f} | Avg Stab: {avg_stability:.2f}')

In [ ]:
# Export to CSV for easy sharing

export_path = os.path.join(config.BASE_DIR, 'logs', 'param_grid_comparison.csv')
os.makedirs(os.path.dirname(export_path), exist_ok=True)
comparison_df.to_csv(export_path, index=False)
print(f'Exported to: {export_path}')